# Level3 PPO Finetuning

This notebook warm-starts a new level3 PPO run from `level3_localobs_relax_final.ckpt`.

Default plan:
- start from the current best local-observation checkpoint;
- use a safer reward profile with more obstacle/crash pressure;
- train short runs first and evaluate every saved checkpoint on seeds 1-100.

Switch `TRAIN_ENV_TOML` to `level3_dr.toml` only when you want train-only DR finetuning.

In [1]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prevent JAX from preallocating most GPU memory before PyTorch asks for tensors.
# This must be set before importing modules that import JAX.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import numpy as np
import torch
import wandb

from lsy_drone_racing.control.ppo_level3_observation import checkpoint_hidden_dim, unpack_checkpoint
from lsy_drone_racing.control.train_CleanRL_ppo_level3 import (
    Args,
    LEVEL3_OBSERVATION_LAYOUT,
    debug_rollout,
    train_ppo,
)

print("repo:", ROOT)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("level3 observation layout:", LEVEL3_OBSERVATION_LAYOUT)

repo: /home/miao/repo/lsy_drone_racing
torch: 2.8.0+cu128
cuda available: True
level3 observation layout: level3_target_gate_nearest_gate_2obs_local_history_v5


W0619 20:42:34.503629  786730 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W0619 20:42:34.506208  786554 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


## Finetune Config

`level3.toml` is the recommended first pass because the current final checkpoint is still crash-heavy. After a safer nominal checkpoint improves, change `TRAIN_ENV_TOML` to `level3_dr.toml` and warm-start from that safer checkpoint.

In [2]:
# TRAIN_ENV_TOML = "level3.toml"
TRAIN_ENV_TOML = "level3_dr.toml"  # enable this for train-only DR finetuning

RUN_NAME = "level3_localobs_safer_finetune_from_final_highcrashpenalty"
INITIAL_MODEL_NAME = "checkpoints/level3_localobs_relax/level3_localobs_relax_final.ckpt"

CONFIG = TRAIN_ENV_TOML
CONFIG_PATH = ROOT / "config" / TRAIN_ENV_TOML
if not CONFIG_PATH.exists():
    raise FileNotFoundError(CONFIG_PATH)

CONTROL_DIR = ROOT / "lsy_drone_racing/control"
INITIAL_MODEL_PATH = CONTROL_DIR / INITIAL_MODEL_NAME
if not INITIAL_MODEL_PATH.exists():
    raise FileNotFoundError(INITIAL_MODEL_PATH)

CHECKPOINT_DIR = CONTROL_DIR / "checkpoints" / RUN_NAME
MODEL_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_final.ckpt"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

JAX_DEVICE = "gpu"
CUDA = True
DEBUG_JAX_DEVICE = "cpu"
DEBUG_TORCH_DEVICE = torch.device("cpu")

WANDB_ENABLED = True
WANDB_PROJECT_NAME = "ADR-PPO-Racing"
WANDB_ENTITY = None
WANDB_MODE = "online"  # use "offline" if needed

NUM_ENVS_DEBUG = 4
NUM_ENVS_TRAIN = 1024
NUM_STEPS = 32
TOTAL_TIMESTEPS_TRAIN = 50_000_000
CHECKPOINT_INTERVAL_STEPS = 10_000_000

LEARNING_RATE = 5e-5
GAMMA = 0.99
GAE_LAMBDA = 0.95
UPDATE_EPOCHS = 5
NUM_MINIBATCHES = 8
ENT_COEF = 0.02
TARGET_KL = 0.03
HIDDEN_DIM = 256

print("config:", CONFIG_PATH)
print("initial:", INITIAL_MODEL_PATH)
print("output:", MODEL_PATH)

config: /home/miao/repo/lsy_drone_racing/config/level3_dr.toml
initial: /home/miao/repo/lsy_drone_racing/lsy_drone_racing/control/checkpoints/level3_localobs_relax/level3_localobs_relax_final.ckpt
output: /home/miao/repo/lsy_drone_racing/lsy_drone_racing/control/checkpoints/level3_localobs_safer_finetune_from_final_highcrashpenalty/level3_localobs_safer_finetune_from_final_highcrashpenalty_final.ckpt


## Reward Profile

This uses the same reward parameters as `notebooks/train_level3_ppo.ipynb`, so the finetune only changes the initial weights and training schedule.

In [3]:
REWARD_KWARGS = dict(
    progress_coef=0.0,
    gate_stage_coef=15.0,
    gate_axis_coef=15.0,
    near_gate_coef=0.0,

    gate_bonus=120.0,
    gate_back_bonus=20.0,
    finish_bonus=160.0,

    missed_gate_penalty=0.0,
    wrong_side_penalty=6.0,
    crash_penalty=100.0,

    obstacle_coef=7.5,
    obstacle_margin=0.35,
    obstacle_clearance_coef=0.0,

    timeout_penalty=80.0,
    time_penalty=0.03,

    act_coef=0.03,
    d_act_th_coef=0.10,
    d_act_xy_coef=0.10,
    cmd_tilt_coef=1.0,
    rpy_coef=1.0,
    tilt_limit_deg=40.0,
    tilt_excess_coef=15.0,
)

REWARD_KWARGS

{'progress_coef': 0.0,
 'gate_stage_coef': 15.0,
 'gate_axis_coef': 15.0,
 'near_gate_coef': 0.0,
 'gate_bonus': 120.0,
 'gate_back_bonus': 20.0,
 'finish_bonus': 160.0,
 'missed_gate_penalty': 0.0,
 'wrong_side_penalty': 6.0,
 'crash_penalty': 100.0,
 'obstacle_coef': 7.5,
 'obstacle_margin': 0.35,
 'obstacle_clearance_coef': 0.0,
 'timeout_penalty': 80.0,
 'time_penalty': 0.03,
 'act_coef': 0.03,
 'd_act_th_coef': 0.1,
 'd_act_xy_coef': 0.1,
 'cmd_tilt_coef': 1.0,
 'rpy_coef': 1.0,
 'tilt_limit_deg': 40.0,
 'tilt_excess_coef': 15.0}

In [4]:
def build_args(**overrides):
    base = dict(
        config=CONFIG,
        total_timesteps=TOTAL_TIMESTEPS_TRAIN,
        num_envs=NUM_ENVS_TRAIN,
        num_steps=NUM_STEPS,
        learning_rate=LEARNING_RATE,
        gamma=GAMMA,
        gae_lambda=GAE_LAMBDA,
        update_epochs=UPDATE_EPOCHS,
        num_minibatches=NUM_MINIBATCHES,
        ent_coef=ENT_COEF,
        target_kl=TARGET_KL,
        hidden_dim=HIDDEN_DIM,
        cuda=CUDA,
        jax_device=JAX_DEVICE,
        wandb_project_name=WANDB_PROJECT_NAME,
        wandb_entity=WANDB_ENTITY,
        n_obs=2,
        **REWARD_KWARGS,
    )
    base.update(overrides)
    return Args.create(**base)

def torch_device(args):
    return torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")

## Check Warm Start

This must print `obs_dim=68`, `hidden_dim=256`, and the same observation layout as the training wrapper.

In [5]:
checkpoint = torch.load(INITIAL_MODEL_PATH, map_location="cpu")
model_state_dict, observation_layout = unpack_checkpoint(checkpoint)
initial_hidden_dim = checkpoint_hidden_dim(checkpoint, model_state_dict)
obs_dim = int(model_state_dict["actor_mean.0.weight"].shape[1])

print("observation_layout:", observation_layout)
print("obs_dim:", obs_dim)
print("hidden_dim:", initial_hidden_dim)

assert observation_layout == LEVEL3_OBSERVATION_LAYOUT
assert obs_dim == 68
assert initial_hidden_dim == HIDDEN_DIM

observation_layout: level3_target_gate_nearest_gate_2obs_local_history_v5
obs_dim: 68
hidden_dim: 256


## W&B

Do not put API keys in the notebook. If needed, run `wandb login` in the terminal or let this cell prompt you.

In [6]:
os.environ["WANDB_MODE"] = WANDB_MODE
os.environ["WANDB_NOTEBOOK_NAME"] = str(ROOT / "notebooks/finetune_level3_ppo.ipynb")

if WANDB_ENABLED:
    if WANDB_MODE != "offline":
        wandb.login()
    print(f"W&B enabled: project={WANDB_PROJECT_NAME}, entity={WANDB_ENTITY}")
else:
    print("W&B disabled")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/miao/.netrc.
wandb: Currently logged in as: xiaochenmiao (aojili77-technical-university-of-munich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B enabled: project=ADR-PPO-Racing, entity=None


## Optional Debug Rollout

Run this before a long job if you changed config or reward coefficients. It uses zero actions and checks that the environment, reward wrapper, and observation shape compile.

In [7]:
RUN_DEBUG_ROLLOUT = True

if RUN_DEBUG_ROLLOUT:
    debug_args = build_args(
        total_timesteps=NUM_ENVS_DEBUG * NUM_STEPS,
        num_envs=NUM_ENVS_DEBUG,
        debug_obs=True,
        debug_reward_every=1,
    )
    print("debug device:", DEBUG_TORCH_DEVICE, "| debug jax_device:", DEBUG_JAX_DEVICE)
    debug_rollout(
        debug_args,
        n_steps=20,
        device=DEBUG_TORCH_DEVICE,
        jax_device=DEBUG_JAX_DEVICE,
    )

debug device: cpu | debug jax_device: cpu


/home/miao/repo/lsy_drone_racing/.pixi/envs/gpu/lib/python3.13/site-packages/jax/_src/abstract_arrays.py:135: RuntimeWarning: overflow encountered in cast
  return literals.TypedNdArray(np.asarray(x, dtype), weak_type=False)


[obs-debug] obs_dim=68 num_envs=4
[obs-debug] pos_z                      slice=000:001 mean= 0.007 std= 0.005 min= 0.002 max= 0.012
[obs-debug] vel_body                   slice=001:004 mean=-0.017 std= 0.053 min=-0.087 max= 0.108
[obs-debug] ang_vel                    slice=004:007 mean= 0.004 std= 0.089 min=-0.123 max= 0.178
[obs-debug] rotmat                     slice=007:016 mean= 0.332 std= 0.472 min=-0.097 max= 0.999
[obs-debug] target_gate_corners_body   slice=016:028 mean= 0.377 std= 0.664 min=-1.007 max= 1.666
[obs-debug] target_gate_known          slice=028:029 mean= 0.250 std= 0.433 min= 0.000 max= 1.000
[obs-debug] nearest_other_gate_corners_body slice=029:041 mean= 0.551 std= 0.623 min=-0.776 max= 1.437
[obs-debug] nearest_other_gate_known   slice=041:042 mean= 0.000 std= 0.000 min= 0.000 max= 0.000
[obs-debug] nearest_obstacles_heading_xy slice=042:050 mean= 0.375 std= 0.495 min=-0.600 max= 1.000
[obs-debug] last_action                slice=050:054 mean= 0.000 std= 0.000 m

## Train

This is a warm start, not a full optimizer resume. The policy/critic weights come from `INITIAL_MODEL_PATH`, while Adam starts fresh at `LEARNING_RATE`.

In [8]:
args = build_args()
device = torch_device(args)

print("device:", device)
print("env:", args.config)
print("total_timesteps:", args.total_timesteps)
print("checkpoint_interval:", CHECKPOINT_INTERVAL_STEPS)
print("initial_model_path:", INITIAL_MODEL_PATH)
print("model_path:", MODEL_PATH)

train_ppo(
    args=args,
    model_path=MODEL_PATH,
    device=device,
    jax_device=JAX_DEVICE,
    wandb_enabled=WANDB_ENABLED,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_interval=CHECKPOINT_INTERVAL_STEPS,
    initial_model_path=INITIAL_MODEL_PATH,
)

device: cuda
env: level3_dr.toml
total_timesteps: 50000000
checkpoint_interval: 10000000
initial_model_path: /home/miao/repo/lsy_drone_racing/lsy_drone_racing/control/checkpoints/level3_localobs_relax/level3_localobs_relax_final.ckpt
model_path: /home/miao/repo/lsy_drone_racing/lsy_drone_racing/control/checkpoints/level3_localobs_safer_finetune_from_final_highcrashpenalty/level3_localobs_safer_finetune_from_final_highcrashpenalty_final.ckpt


Training on device: cuda | Environment device: gpu


/home/miao/repo/lsy_drone_racing/.pixi/envs/gpu/lib/python3.13/site-packages/jax/_src/abstract_arrays.py:135: RuntimeWarning: overflow encountered in cast
  return literals.TypedNdArray(np.asarray(x, dtype), weak_type=False)


initialized agent weights from /home/miao/repo/lsy_drone_racing/lsy_drone_racing/control/checkpoints/level3_localobs_relax/level3_localobs_relax_final.ckpt; optimizer starts fresh
Iter 1/1525 took 17.81 seconds
Iter 2/1525 took 3.02 seconds
Iter 3/1525 took 2.90 seconds
Iter 4/1525 took 2.88 seconds
Iter 5/1525 took 2.90 seconds
Iter 6/1525 took 2.67 seconds
Iter 7/1525 took 2.63 seconds
Iter 8/1525 took 2.69 seconds
Iter 9/1525 took 2.82 seconds
Iter 10/1525 took 2.86 seconds
Iter 11/1525 took 2.65 seconds
Iter 12/1525 took 2.71 seconds
Iter 13/1525 took 3.27 seconds
Iter 14/1525 took 2.98 seconds
Iter 15/1525 took 2.95 seconds
Iter 16/1525 took 2.94 seconds
Iter 17/1525 took 2.91 seconds
Iter 18/1525 took 2.88 seconds
Iter 19/1525 took 3.20 seconds
Iter 20/1525 took 3.40 seconds
Iter 21/1525 took 3.23 seconds
Iter 22/1525 took 3.32 seconds
Iter 23/1525 took 3.44 seconds
Iter 24/1525 took 3.02 seconds
Iter 25/1525 took 3.10 seconds
Iter 26/1525 took 3.49 seconds
Iter 27/1525 took 3.26

[-104.35858917236328,
 -104.23145294189453,
 -103.37641906738281,
 -103.58562469482422,
 -103.83306884765625,
 -109.61088562011719,
 -103.47146606445312,
 -100.35469818115234,
 -103.60011291503906,
 -104.19074249267578,
 -103.10198211669922,
 -98.09015655517578,
 -104.00189208984375,
 -101.84979248046875,
 -103.60580444335938,
 -104.0103530883789,
 -104.11544036865234,
 -109.80084228515625,
 -103.92707824707031,
 -103.71820831298828,
 -103.36714935302734,
 -104.11225128173828,
 -104.41667938232422,
 -110.02372741699219,
 -103.19622039794922,
 -103.80583190917969,
 -103.22415161132812,
 -110.41151428222656,
 -104.72351837158203,
 -103.43336486816406,
 -104.16332244873047,
 -106.06855010986328,
 -104.63508605957031,
 -103.39399719238281,
 -104.86363220214844,
 -110.27595520019531,
 -109.56843566894531,
 -114.67284393310547,
 -188.190185546875,
 -119.17916107177734,
 -185.61183166503906,
 -118.7730941772461,
 -189.2821044921875,
 -187.19235229492188,
 -113.0838851928711,
 -116.40658569335

## After Training

Evaluate each saved checkpoint with the same deterministic seed sweep used before: seeds `1-100`, `config/level3.toml`, and the inference controller path. Compare against the current baseline:

- `level3_localobs_relax_final`: success `15/100`, crash `85/100`, mean gates `1.38`, mean success time `6.40s`.

A checkpoint is worth keeping only if success increases without moving most failures from gate frames into obstacles.